In [ ]:
import numpy as np
from shapely.geometry import Polygon
import overturemaps
import geopandas as gpd
import pandas as pd

from IPython.display import IFrame

import gc
import dask_geopandas as geodask
import dask
import pyarrow.parquet as pq

import plotly.express as px
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
#from palettable.colorbrewer.sequential import Oranges_9, PuRd_9, YlGnBu_9, PuBuGn_9, YlOrRd_9, RdPu_7
#from palettable.colorbrewer.qualitative import Paired_12 
#from palettable.colorbrewer.diverging import RdYlGn_7, Spectral_9, PiYG_6

from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap, apply_categorical_cmap
import pydeck

import h3pandas
import h3

import datashader as ds
import datashader.transfer_functions as ds_function
from datashader.utils import export_image
from functools import partial
import colorcet as cc

import os, shutil, glob
import time
import warnings
warnings.simplefilter("ignore")


In [ ]:
# results folder
if not os.path.exists('output'):
    os.makedirs('output')

scratch = 'scratch'

In [ ]:
fin_bbox = 19.84, 59.68, 31.78, 70.25

In [ ]:
%%time

table = overturemaps.record_batch_reader("place", fin_bbox).read_all()

# Temporarily required as of Lonboard 0.8 to avoid a Lonboard bug
table = table.combine_chunks()

In [ ]:
!wget -N https://a3s.fi/swift/v1/AUTH_a6b8530017f34af9861fcf45a738ad3f/L2-CellTowers/L2-CountryAdmin-data.gpkg

In [ ]:
# read country layer
world_admin = gpd.read_file('data/L2-CountryAdmin-data.gpkg')

In [ ]:
# clean and get needed columns
world_admin = world_admin.dropna()

world_gdf = world_admin[['name', 'continent', 'geometry']]

world_gdf.head(3)

In [ ]:
# get Finland border
fin = world_gdf.loc[world_gdf['name']=='Finland']

In [ ]:
ax = fin.plot();

ax.axis('off');

In [ ]:
# to GeoDataFrame
geodata = gpd.GeoDataFrame(table.to_pandas(), 
                           geometry=gpd.GeoSeries.from_wkb(table['geometry'])
                          )

In [ ]:
# Clip data to Finland
fin_pois = gpd.clip(geodata, fin)

In [ ]:
fin_pois.head(3) 

In [ ]:
len(fin_pois)


In [ ]:
# to Geoparquet
#fin_pois.to_parquet('output/pois_finland.parquet', index=False)

In [ ]:
YlGnBu_9.mpl_colormap

In [ ]:
# apply the colour palette

confidence_array = fin_pois['confidence'].to_numpy()

colors = apply_continuous_cmap(confidence_array, YlGnBu_9)

In [ ]:
# ---- Create a Layer

layer = ScatterplotLayer.from_geopandas(
                                    # --- Select only a few attribute columns from the table
                
                                    fin_pois[["id", "categories", "geometry", "confidence"]],
                                    get_radius = np.array([5*value for value in fin_pois['confidence'].to_numpy()]),
                                    get_fill_color=colors,
                                )

In [ ]:
view_state = {
    "longitude": 26.814052,
    "latitude": 62.399194,
    "zoom": 6,
    "pitch": 40,
    "bearing": 4,
}
m = Map(layer, view_state=view_state)

In [ ]:
# Extract `primary` and `alternate` columns
def extract_categories(cat):
    if isinstance(cat, dict):
        primary = cat.get('primary', None)
        alternate = cat.get('alternate', [])
        if not isinstance(alternate, list):
            alternate = []
    else:
        primary = None
        alternate = []
    # pad alternates to length 2
    alternate = (alternate + [None, None])[:2]
    return pd.Series([primary] + alternate)

fin_pois[['primary', 'alternate_1', 'alternate_2']] = fin_pois['categories'].apply(extract_categories)



In [ ]:
import numpy as np

def extract_categories(cat):
    if isinstance(cat, dict):
        primary = cat.get('primary', None)
        alternate = cat.get('alternate', None)

        alt1, alt2 = None, None
        if isinstance(alternate, np.ndarray):
            if len(alternate) > 0:
                alt1 = alternate[0]
            if len(alternate) > 1:
                alt2 = alternate[1]
        elif isinstance(alternate, list):
            if len(alternate) > 0:
                alt1 = alternate[0]
            if len(alternate) > 1:
                alt2 = alternate[1]

        return pd.Series([primary, alt1, alt2])
    else:
        return pd.Series([None, None, None])

fin_pois[['primary', 'alternate_1', 'alternate_2']] = fin_pois['categories'].apply(extract_categories)


In [ ]:
fin_pois.tail(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
fin_pois['confidence'].hist(bins=30, color='steelblue', edgecolor='black')
plt.title("Histogram of confidence column", fontsize=14)
plt.xlabel("Confidence", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Count the frequency
primary_counts = fin_pois['primary'].value_counts()

In [ ]:
primary_counts.head(20)

In [ ]:
# Step 2: Filter rows where primary category == community_services_non_profits
community_df = fin_pois[fin_pois['primary'] == 'community_services_non_profits']

# Step 3: From those, extract the *alternate* categories (if any)
community_df['alternate_1'] = community_df['categories'].apply(
    lambda x: x['alternate'][0] if isinstance(x, dict) and isinstance(x.get('alternate'), (list, np.ndarray)) and len(x['alternate']) > 0 else None
)

# Step 4: Count frequencies of the first alternate category
alt_counts = community_df['alternate_1'].value_counts().head(15)

# Step 5: Plot
plt.figure(figsize=(10,6))
alt_counts.plot(kind='bar', color='coral', edgecolor='black')
plt.title("Top 10 Alternate Categories in Community Services Non-Profits", fontsize=14)
plt.xlabel("Alternate Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
geo_hsk = gpd.read_file("data/hsk_shapefile/Helsinki_region_municipality.shp")

In [ ]:
fin_pois = fin_pois.set_crs("EPSG:4326", inplace=False)
geo_hsk = geo_hsk.set_crs("EPSG:4326", inplace=False)

In [ ]:

# Ensure both GeoDataFrames are in the same CRS
if fin_pois.crs != geo_hsk.crs:
    geo_hsk = geo_hsk.to_crs(fin_pois.crs)

# Spatial join or mask
fin_pois_hsk = gpd.sjoin(fin_pois, geo_hsk, how="inner", predicate="within")

In [ ]:
# Count the frequency
primary_counts_hsk = fin_pois_hsk['primary'].value_counts()

In [ ]:
fin_pois_hsk.shape

In [ ]:
fin_pois_hsk = fin_pois_hsk[fin_pois_hsk["confidence"] >= 0.50] #29.3K

In [ ]:
n_unique_categories = fin_pois_hsk['primary'].nunique()
print(f"Unique primary categories: {n_unique_categories}")

In [ ]:
category_counts = (
    fin_pois_hsk['primary']
    .value_counts(dropna=False)
    .reset_index(name='count')
    .rename(columns={'index': 'category'})
)

print(category_counts.head())


In [ ]:
category_counts.to_csv("data/category_list.csv")

In [ ]:
import json

with open("./data/category_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

In [ ]:
# Build reverse map: value → category
reverse_map = {v: k for k, vals in mapping.items() for v in vals}

category_counts["category"] = category_counts["primary"].map(reverse_map)

In [ ]:
fin_pois_hsk["category"] = fin_pois_hsk["primary"].map(reverse_map)

In [ ]:
fin_pois_hsk[fin_pois_hsk["category"].isna()]

In [ ]:
import geopandas as gpd
import folium
from branca.colormap import linear

# Ensure CRS is WGS84
if fin_pois_hsk.crs != "EPSG:4326":
    fin_pois_hsk = fin_pois_hsk.to_crs(epsg=4326)

# Unique categories and color map
categories = sorted(fin_pois_hsk['category'].dropna().unique())
colors = linear.Set1_09.scale(0, len(categories)).to_step(n=len(categories))
color_map = {cat: colors.rgb_hex_str(i) for i, cat in enumerate(categories)}

# Base map
center = [
    fin_pois_hsk.geometry.y.mean(),
    fin_pois_hsk.geometry.x.mean()
]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

# Add base layers
folium.TileLayer('OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter').add_to(m)

# Add points
for idx, row in fin_pois_hsk.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=color_map.get(row['category'], 'gray'),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(str(row['category']), parse_html=True)
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = """
<div style='position: fixed; bottom: 50px; left: 50px; width: 200px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;'>
&nbsp;<b>Categories</b><br>
"""
for cat, col in color_map.items():
    legend_html += f"&nbsp;<i style='background:{col}'>&nbsp;&nbsp;&nbsp;&nbsp;</i> {cat}<br>"
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

m.save("pois_by_category_map_no_cluster.html")


In [ ]:
# Create H3 hexagons (resolution 9)
def polyfill_geometry(geom, res):
    return list(h3.polyfill(geom.__geo_interface__, res, geo_json_conformant=True))

hex_ids = set()
for geom in geo_hsk.geometry:
    hex_ids.update(polyfill_geometry(geom, 9))

# Build GeoDataFrame of hexagons
h3_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]
h3_hsk = gpd.GeoDataFrame({'h3_id': list(hex_ids), 'geometry': h3_geoms}, crs='EPSG:4326')



In [ ]:

# Check for conflicting column before join
if 'index_right' in fin_pois_hsk.columns:
    fin_pois_hsk = fin_pois_hsk.drop(columns='index_right')

In [ ]:
# Spatial join with POIs (keeping only those in fin_pois_hsk)
pois_with_h3 = gpd.sjoin(fin_pois_hsk, h3_hsk, predicate="intersects", how="left")



In [ ]:
# Group by h3_id and category, then count how many POIs per category per hexagon
category_hsk_freq = pois_with_h3.groupby(["h3_id", "category"]).size().reset_index(name="count")

In [ ]:
category_hsk_freq.category.unique()

In [ ]:
category_hsk_freq.to_parquet("data/pois_per_hex.parquet")

In [ ]:
category_hsk_freq = pd.read_parquet("data/pois_per_hex.parquet")

In [ ]:

from mapclassify import NaturalBreaks

def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

# Filter by target category
target_category = "Grocery Stores & Supermarkets"
category_filtered = category_hsk_freq[category_hsk_freq['category'] == target_category].copy()

# Build geometry from H3 codes
category_filtered["geometry"] = category_filtered["h3_id"].apply(h3_to_polygon)

# Convertir a GeoDataFrame
category_gdf = gpd.GeoDataFrame(category_filtered, geometry="geometry", crs="EPSG:4326")


In [ ]:
# Classify with natural breaks
nb = NaturalBreaks(y=category_gdf["count"], k=5)
category_gdf["nb_class"] = nb.yb

# Prepare legend labels with the actual class ranges
labels = []
for i in range(nb.k):
    lower = int(nb.bins[i-1]) + 1 if i > 0 else 0
    upper = int(nb.bins[i])
    labels.append(f"{lower} - {upper}")

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors



# 1. Reproject to EPSG:3857 for contextily
category_gdf_3857 = category_gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# 2. Plot hexagons with transparency (EPSG:3857)
category_gdf_3857.plot(
    column="nb_class",
    ax=ax,
    cmap="OrRd",
    alpha=0.6,            # hexagon transparency
    edgecolor="black",
    linewidth=0.2,
    legend=False
)

# 3. Set the axis limits to the GeoDataFrame bounding box to zoom in on Helsinki
minx, miny, maxx, maxy = category_gdf_3857.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# 4. Add a semi-transparent basemap
ctx.add_basemap(ax, source=basemaps.POSITRON, alpha=0.7, zoom=12)

# 5. Build the legend with solid colours (no alpha)
labels = []
for i in range(nb.k):
    lower = int(nb.bins[i-1]) + 1 if i > 0 else 0
    upper = int(nb.bins[i])
    labels.append(f"{lower} - {upper}")

cmap = plt.cm.OrRd
colors = [cmap(i / (nb.k - 1)) for i in range(nb.k)]
patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(nb.k)]

plt.legend(handles=patches, title="Frequency ranges", loc='lower left')

ax.set_title(f"Frequency of '{target_category}' by H3 Cell (Natural Breaks)")
ax.axis("off")
plt.tight_layout()
plt.show()



In [ ]:
category_gdf

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import contextily as ctx; import basemaps
from shapely.geometry import Polygon
from mapclassify import NaturalBreaks
import h3
import os

# Asegurar carpeta de salida
os.makedirs("output", exist_ok=True)

# --- Funciones auxiliares ---
def h3_to_polygon(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

def classify_and_plot(ax, category_df, category_name, bounds, cmap="OrRd"):
    # Build geometry
    category_df = category_df.copy()
    category_df["geometry"] = category_df["h3_id"].apply(h3_to_polygon)
    gdf = gpd.GeoDataFrame(category_df, geometry="geometry", crs="EPSG:4326").to_crs(epsg=3857)

    # Clasificar
    nb = NaturalBreaks(y=gdf["count"], k=5)
    gdf["nb_class"] = nb.yb

    # Plot
    gdf.plot(
        column="nb_class",
        ax=ax,
        cmap=cmap,
        alpha=0.6,
        edgecolor="black",
        linewidth=0.2,
        legend=False
    )

    ax.set_xlim(*bounds[:2])
    ax.set_ylim(*bounds[2:])
    ctx.add_basemap(ax, source=basemaps.POSITRON, alpha=0.7, zoom=12)

    ax.set_title(category_name, fontsize=10, pad=8)
    ax.axis("off")

# --- Prepare data ---
target_categories = [
    "Restaurant & Entertainment",
    "Well-being & Lifestyle",
    "Jobs, Professional Services & Religious",
    "Educational Facilities",
    "Shopping & Retail",
    "Grocery Stores & Supermarkets"
]

# Filter by these categories
category_filtered_all = category_hsk_freq[category_hsk_freq["category"].isin(target_categories)].copy()
category_filtered_all["geometry"] = category_filtered_all["h3_id"].apply(h3_to_polygon)
category_all_gdf = gpd.GeoDataFrame(category_filtered_all, geometry="geometry", crs="EPSG:4326").to_crs(epsg=3857)

# Common bounding box
minx, miny, maxx, maxy = category_all_gdf.total_bounds
bounds = (minx, maxx, miny, maxy)

# --- Plot ---
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 15))
axes = axes.flatten()

for i, cat in enumerate(target_categories):
    cat_df = category_hsk_freq[category_hsk_freq["category"] == cat].copy()
    classify_and_plot(axes[i], cat_df, cat, bounds)

# Ajustes de layout
plt.tight_layout(h_pad=3)  # espacio vertical entre subplots
plt.savefig("output/h3_category_frequency_grid.png", dpi=300)
plt.show()

